# Structured Output
1. Pydantic（推荐）
2. TypeDict

## with_structured_output()
- 把结构体转成结构化约束（比如：`JSON Schema`），把返回解析为结构体
- 默认走`mode='json_schmea'`,大致报文结构（response_format）：
    ```json
      {
          "model": "qwen3.7-plus",
          "messages": [
            {
              "role": "user",
              "content": "请严格解析为JSON: 张三今年30岁已经转成AI架构师了"
            }
          ],
          "response_format": {
            "type": "json_schema",
            "json_schema": {
              "name": "Person",
              "schema": {
                "type": "object",
                "description": "个人信息结构数据",
                "properties": {
                  "name": { "type": "string", "description": "姓名" },
                  "age": { "type": "integer", "description": "年龄", "maximum": 150 },
                  "occu": { "type": "string", "description": "职位" }
                },
                "required": ["name", "age", "occu"]
            }
          }
      }
    }
    ```

In [28]:
from typing import Optional

from common import init_simple_dashscope_model
from pydantic import BaseModel, Field
from rich import print as rprint

# model = init_simple_dashscope_model('qwen3.7-plus').bind(
#     extra_body={"enable_thinking": False}
# )
model = init_simple_dashscope_model('qwen-max')


## Pydantic Structured Output - JSON Schema（默认）
- qwen-max不支持,且在绑定了结构体调用时，提示词必须出现`json`字样，否则报400,
- qwen3.7-plus支持，但需结合提示词`严格解析为JSON`辅助，解析更稳定。如果想保持提示词简洁，结构体的字段名要清晰，不要有歧义。比如`Person`的`user_name`能够很好的解析，但是如果是`name`，模型总是解析成了一个工具名

In [13]:
class Person(BaseModel):
    """个人信息结构数据"""

    # 不用name，qwen3.7-plus会解析为其他的值
    user_name: str = Field(
        description="人名"
    )
    age: int = Field(
        description="年龄",
        le=150
    )
    occu: str = Field(
        description="职位"
    )

model_with_structured = model.with_structured_output(Person)
# resp = model_with_structured.invoke('请严格解析为JSON: 张三今年30岁已经转成AI架构师了')
resp = model_with_structured.invoke('张三今年30岁已经转成AI架构师了')

rprint(resp)

Person(user_name='张三', age=30, occu='AI架构师')

## Pydantic Structured Output - Function Calling
- qwen-max支持
- qwen3.7-plus默认开启了thinking,设置了`method=function_calling`后，本质是通过`tools+tool_choice`强制使用工具，而tool_choice和thinking不能共存。可以尝试关闭深度思考。

In [29]:
class Person(BaseModel):
    """个人信息结构数据"""

    # 不用name，会解析为其他的值
    user_name: str = Field(
        description="人名"
    )
    age: int = Field(
        description="年龄",
        le=150
    )
    occu: str = Field(
        description="职位"
    )

model_with_structured = model.with_structured_output(schema=Person, method='function_calling')
# resp = model_with_structured.invoke('请严格解析为JSON: 张三今年30岁已经转成AI架构师了')
resp = model_with_structured.invoke('张三今年30岁已经转成AI架构师了')

rprint(resp)

Person(user_name='张三', age=30, occu='AI架构师')

## Pydantic结构化-高级特性
- 可选字段。当不存在值时，int型最终默认显示0，字符串默认""，设置为`Optinal[Type]`，当值不存在，会显示None。如果之前已经解析出了结果，真实不存在字段值的情况下，模型会产生幻觉。而设置为可选，值缺失会设置为None，更符合实际情况。
- 默认值。 `Field`的`default`设置默认值
- 枚举。`Literal`或者枚举类
- 列表
- 嵌套
- 限制

### 可选字段

In [37]:
from typing import Optional
from enum import Enum

class Live(Enum):
    BAO_AN = 


class Person(BaseModel):
    user_name: str = Field(
        description="姓名"
    )
    age: Optional[int] = Field(
        description="年龄"
    )
    gender: str = Field(
        default="男",
        description="性别"
    )
    occu: str = Field(
        description="职位"
    )

model_with_structured = model.with_structured_output(Person, method="function_calling")
resp = model_with_structured.invoke("除了张三，还有人自己创业，成为了董事长")
rprint(resp)

Person(user_name='李四', age=35, gender='男', occu='董事长')